## Setup

In [ ]:
!pip install numpy
!pip install pandas
!pip install statsmodels
!pip install pyBigWig

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pyliftover import LiftOver
import pyBigWig
import warnings
from tqdm import tqdm

# Suppress all warnings
warnings.filterwarnings("ignore")

from matplotlib.colors import LinearSegmentedColormap
import mpl_scatter_density

# "Viridis-like" colormap with white background
white_viridis = LinearSegmentedColormap.from_list('white_viridis', [
    (0, '#ffffff'),
    (1e-20, '#440053'),
    (0.2, '#404388'),
    (0.4, '#2a788e'),
    (0.6, '#21a784'),
    (0.8, '#78d151'),
    (1, '#fde624'),
], N=256)

In [4]:
# Specify project directories
data_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/'
results_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/osthoag/wgs-constraint-llm/results/'

# Specify the file paths
annotation_file_path = data_path + 'gencode.v44.basic.annotation.gtf.gz'
constraint_file_path = data_path + 'gnomad.v4.0.constraint_metrics.tsv'
gerp_file_path = data_path + 'All_hg38_RS.bw'
alpha_missense_file_path = data_path + 'AlphaMissense_hg38.tsv.gz'
hmm_predictions_path = results_path + "HMM_rgc_0.9_over20_chr2_predictions_rgc_wes.tsv.gz"

## Load data

In [ ]:
# GERP RS annotation.
#
# This cell previously inlined a bigWig lookup that indexed a 0-based value
# vector with 1-based HMM positions, shifting every score one base downstream.
# The lookup now lives in src/wgs_constraint/gerp.py, so the three notebooks
# that need it cannot drift apart again; the coordinate convention is an
# explicit argument there rather than an assumption.
#
# Loading the predictions is kept here because later cells use `pred`.
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from wgs_constraint import annotate_with_gerp

# --- load HMM predictions (once) ---
pred = pd.read_csv(hmm_predictions_path, sep="\t", dtype={"chr": "string"})
if "position" in pred.columns and "pos" not in pred.columns:
    pred = pred.rename(columns={"position": "pos"})
pred["pos"] = pred["pos"].astype(np.int64)
pred["chr"] = pred["chr"].astype("string")
pred_cols = pred.columns.tolist()

merged_df = annotate_with_gerp(
    pred,
    gerp_file_path,
    position_base=1,      # HMM positions are 1-based
)
merged_df["chr"] = merged_df["chr"].astype("category")
merged_df


In [ ]:
# Get predictions from HMM and GERP
predictions_df = pd.read_csv("HMM_rgc_ALL_RS_merged_predictions.tsv.gz", sep='\t')
predictions_df

In [ ]:
# Read the file into a pandas DataFrame
# constraint_df = pd.read_csv(constraint_file_path, sep='\t', usecols=['gene', 'transcript', 'mane_select', 'mis.z_score'])
constraint_df = pd.read_csv(constraint_file_path, sep='\t')

# # Compute the MTR
constraint_df['MTR'] = (constraint_df['mis.obs'] / (constraint_df['mis.obs'] + constraint_df['syn.obs'])) / (constraint_df['mis.exp'] / (constraint_df['mis.exp'] + constraint_df['syn.exp']))

# Drop unnecessary columns
# constraint_df = constraint_df[['gene', 'transcript', 'mane_select', 'mis.z_score', 'lof.z_score', 'MTR']]

constraint_df

In [ ]:
# Read the GTF file into a pandas DataFrame
gene_df = pd.read_csv(annotation_file_path, sep='\t', comment='#', header=None, 
                      names=['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute'], 
                      dtype={'start': int, 'end': int})

# Extract 'gene_id' from attributes
gene_df['gene_id'] = gene_df['attribute'].str.extract(r'gene_id "(.*?)"')

# Extract 'gene_type' from attributes
gene_df['gene_type'] = gene_df['attribute'].str.extract(r'gene_type "(.*?)"')

# Extract 'gene_name' from attributes
gene_df['gene_name'] = gene_df['attribute'].str.extract(r'gene_name "(.*?)"')

# Extract 'transcript_id' from attributes
gene_df['transcript_id'] = gene_df['attribute'].str.extract(r'transcript_id "(.*?)"')

# Extract 'transcript' and 'num' from transcript_id
gene_df[['transcript', 'transcript_num']] = gene_df['transcript_id'].str.split('.', expand=True)

# Extract 'transcript_name' from attributes
gene_df['transcript_name'] = gene_df['attribute'].str.extract(r'transcript_name "(.*?)"')

# Drop the original attribute column
gene_df = gene_df.drop('attribute', axis=1)

# Filter rows for protein-coding regions
gene_df = gene_df[(gene_df['gene_type'] == 'protein_coding') & (gene_df['feature'] == 'CDS')]

gene_df

In [ ]:
gene_constraint_df = pd.merge(gene_df, constraint_df, left_on=['gene_name', 'transcript'], right_on=['gene', 'transcript'], how='inner')
gene_constraint_df['length'] = abs(gene_constraint_df['end'] - gene_constraint_df['start']) + 1
gene_constraint_df

In [ ]:
# === INSPECT: do we have position-/region-level MTR fields? ===
import pandas as pd
import numpy as np

def _colhits(df, patterns):
    pats = [p.lower() for p in patterns]
    return [c for c in df.columns if any(p in c.lower() for p in pats)]

print("constraint_df shape:", constraint_df.shape)
print("\nconstraint_df candidate position/region columns:")
patterns = [
    "pos", "position", "start", "end", "window", "interval",
    "cds", "coding", "protein", "aa", "amino", "residue",
    "transcript_position", "transcript_pos", "cdna", "codon"
]
hits = _colhits(constraint_df, patterns)
print(hits[:200])
if len(hits) > 200:
    print(f"... plus {len(hits)-200} more")

print("\nconstraint_df key columns present?")
key_cols = ["gene", "transcript", "mis.obs", "syn.obs", "mis.exp", "syn.exp", "MTR", "mis.z_score"]
for c in key_cols:
    print(f"  {c}: {'YES' if c in constraint_df.columns else 'no'}")

print("\nSample rows of constraint_df (selected cols):")
show = [c for c in (["gene","transcript"] + hits[:25] + ["mis.obs","syn.obs","mis.exp","syn.exp","MTR","mis.z_score"]) if c in constraint_df.columns]
display(constraint_df[show].head(5))

print("\nunique transcript count:", constraint_df["transcript"].nunique() if "transcript" in constraint_df.columns else "NA")
print("unique gene count:", constraint_df["gene"].nunique() if "gene" in constraint_df.columns else "NA")

# Also check predictions_df for columns (we'll use chr/pos/prob_0 for codons)
print("\npredictions_df columns:", list(predictions_df.columns))
print("predictions_df shape:", predictions_df.shape)
display(predictions_df.head(3))

# Check GTF CDS frame availability
print("\ngene_df columns:", list(gene_df.columns))
print("gene_df['frame'] unique (first 20):", pd.unique(gene_df["frame"])[:20])
display(gene_df.head(3))

## Evaluate relationship between different gnomAD and HMM constraint metrics

1. Aggregate HMM constraint predictions over each gene
2. Compare to an existing constraint metric from gnomAD using a GLM
3. Plot the relationship

### HMM vs GERP

In [ ]:
merged_df.dropna(subset=["GERP_RS"], inplace=True)

# Define X and y
X = merged_df["prob_0"]
y = merged_df["GERP_RS"]

# Add constant term for intercept
X = sm.add_constant(X)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print R-squared
print("R-squared:", model.rsquared)

# Optionally print full summary
print(model.summary())

In [ ]:
results = []
for chrom, df_chrom in merged_df.groupby("chr"):
    if len(df_chrom) < 10:
        continue
    X = sm.add_constant(df_chrom["prob_0"])
    y = df_chrom["GERP_RS"]
    model = sm.OLS(y, X).fit()
    results.append({
        "chr": chrom,
        "n": len(df_chrom),
        "r_squared": model.rsquared
    })

per_chrom_results = pd.DataFrame(results)
print(per_chrom_results)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import gridspec

# Load the data
predictions_df = pd.read_csv("HMM_rgc_ALL_RS_merged_predictions.tsv.gz", sep='\t')
predictions_df = predictions_df.dropna(subset=['prob_0', 'GERP_RS'])

# Define bins
x_bins = np.arange(0, 1.1, 0.1)  # prob_0 bins of width 0.1
y_min = int(np.floor(predictions_df['GERP_RS'].min()))
y_max = int(np.ceil(predictions_df['GERP_RS'].max()))
y_bins = np.arange(y_min, y_max + 1, 1)  # GERP_RS bins of width 1

# Set up joint plot layout using GridSpec
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2, width_ratios=[4, 1], height_ratios=[1, 4],
                       wspace=0.05, hspace=0.05)

ax_joint = fig.add_subplot(gs[1, 0])
ax_marg_x = fig.add_subplot(gs[0, 0], sharex=ax_joint)
ax_marg_y = fig.add_subplot(gs[1, 1], sharey=ax_joint)

# Joint 2D histogram
hist = ax_joint.hist2d(
    predictions_df['prob_0'],
    predictions_df['GERP_RS'],
    bins=[x_bins, y_bins],
    norm=LogNorm(),
    cmap='Blues'
)

# Marginal histograms
ax_marg_x.hist(predictions_df['prob_0'], bins=x_bins, color='gray')
ax_marg_y.hist(predictions_df['GERP_RS'], bins=y_bins, orientation='horizontal', color='gray')

# Hide tick labels for marginal plots
ax_marg_x.tick_params(axis='x', labelbottom=False)
ax_marg_y.tick_params(axis='y', labelleft=False)

# Labels and title
ax_joint.set_xlabel('HMM Predicted Constraint Probability (prob_0)', fontsize=12)
ax_joint.set_ylabel('GERP RS Score', fontsize=12)
ax_joint.set_title('Joint and Marginal Distributions', fontsize=14, fontweight='bold', pad=20)

# Colorbar
cb = plt.colorbar(hist[3], ax=ax_joint, pad=0.01)
cb.set_label('Log-scaled Count')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Load HMM+GERP predictions
predictions_df = pd.read_csv("HMM_rgc_ALL_RS_merged_predictions.tsv.gz", sep='\t')

# Drop missing values
predictions_df = predictions_df.dropna(subset=['prob_0', 'GERP_RS'])

# Define bin edges
prob_bins = np.arange(0, 1.1, 0.1)     # HMM constraint (prob_0) in bins of 0.1
gerp_bins = np.arange(-12, 7, 1)       # GERP RS in bins of width 1

# Bin the data
predictions_df['prob_0_bin'] = pd.cut(predictions_df['prob_0'], bins=prob_bins, precision=2)
predictions_df['gerp_bin'] = pd.cut(predictions_df['GERP_RS'], bins=gerp_bins, precision=2)

# Marginal distributions
marginal_prob = predictions_df['prob_0_bin'].value_counts(normalize=True).sort_index()
marginal_gerp = predictions_df['gerp_bin'].value_counts(normalize=True).sort_index()

# Expected joint under independence
expected_joint = np.outer(marginal_prob, marginal_gerp) * len(predictions_df)

# Actual joint distribution
actual_joint = predictions_df[['prob_0_bin', 'gerp_bin']].value_counts().sort_index()
actual_joint = np.array([
    [actual_joint.get((prob_bin, gerp_bin), 0) for gerp_bin in marginal_gerp.index]
    for prob_bin in marginal_prob.index
])

# Percent difference and chi-square
percent_diff = (actual_joint - expected_joint) / expected_joint
percent_diff = np.nan_to_num(percent_diff)

chi_sqr = (expected_joint - actual_joint) ** 2 / expected_joint
chi_sqr = np.nan_to_num(chi_sqr)

# Plot
fig, axs = plt.subplots(2, 2, figsize=(12, 12))

# Expected
sns.heatmap(expected_joint, cmap='Blues', norm=LogNorm(), cbar=True,
            xticklabels=marginal_gerp.index, yticklabels=marginal_prob.index, ax=axs[0, 0])
axs[0, 0].set_title('Expected Joint Distribution')
axs[0, 0].set_xlabel('GERP RS')
axs[0, 0].set_ylabel('HMM Constraint (prob_0)')
axs[0, 0].invert_yaxis()

# Actual
sns.heatmap(actual_joint, cmap='Blues', norm=LogNorm(), cbar=True,
            xticklabels=marginal_gerp.index, yticklabels=marginal_prob.index, ax=axs[0, 1])
axs[0, 1].set_title('Actual Joint Distribution')
axs[0, 1].set_xlabel('GERP RS')
axs[0, 1].set_ylabel('HMM Constraint (prob_0)')
axs[0, 1].invert_yaxis()

# Percent diff
sns.heatmap(percent_diff, cmap='coolwarm', center=0, cbar=True,
            xticklabels=marginal_gerp.index, yticklabels=marginal_prob.index, ax=axs[1, 0])
axs[1, 0].set_title('Percent Difference')
axs[1, 0].set_xlabel('GERP RS')
axs[1, 0].set_ylabel('HMM Constraint (prob_0)')
axs[1, 0].invert_yaxis()

# Chi-squared
sns.heatmap(chi_sqr, cmap='OrRd', cbar=True,
            xticklabels=marginal_gerp.index, yticklabels=marginal_prob.index, ax=axs[1, 1])
axs[1, 1].set_title('Chi-Squared Statistic')
axs[1, 1].set_xlabel('GERP RS')
axs[1, 1].set_ylabel('HMM Constraint (prob_0)')
axs[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# (Assuming you already have predictions_df, binning, and chi_sqr calculated...)
# Create figure
plt.figure(figsize=(8, 6))

# Plot chi-squared statistic with log scale
sns.heatmap(
    percent_diff,
#     norm=LogNorm(vmin=1e-1, vmax=np.max(chi_sqr)),  # Log scale with reasonable floor
    xticklabels=marginal_gerp.index,
    yticklabels=marginal_prob.index,
    cmap='coolwarm',
    center=0,
    cbar=True
)

# Axis labels
plt.xlabel('GERP RS Score', fontsize=12)
plt.ylabel('HMM Constraint Probability', fontsize=12)
plt.title('Percent Difference for Observed vs Expected Joint Distribution', fontsize=14, fontweight='bold')

# Optional: invert y-axis to match previous convention
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# Create figure
plt.figure(figsize=(8, 6))

# Plot chi-squared statistic with log scale
sns.heatmap(
    chi_sqr,
    cmap='OrRd',
#     norm=LogNorm(vmin=1e-1, vmax=np.max(chi_sqr)),  # Log scale with reasonable floor
    xticklabels=marginal_gerp.index,
    yticklabels=marginal_prob.index,
    cbar=True
)

# Axis labels
plt.xlabel('GERP RS Score', fontsize=12)
plt.ylabel('HMM Constraint Probability', fontsize=12)
plt.title('Chi-Squared Statistic for Observed vs Expected Joint Distribution', fontsize=14, fontweight='bold')

# Optional: invert y-axis to match previous convention
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:
xticklabels = [f'{i:.1f}' for i in np.arange(0, 1.1, 0.1)]
yticklabels = [f'{i:.1f}' for i in np.arange(0, 1.1, 0.1)]

fig, ax1 = plt.subplots(1, figsize=(7, 6))

# Plot actual joint distribution with log scale
ax1 = sns.heatmap(actual_joint, cmap='Blues', annot=False, fmt=".3f", cbar=True, norm=LogNorm(), ax=ax1)
ax1.set_title('Observed Joint Distribution', fontsize=14)
ax1.set_xlabel('RGC Constraint Probability', fontsize=12)
ax1.set_ylabel('AoU Constraint Probability', fontsize=12)
ax1.invert_yaxis()
ax1.set_xticks(range(11))
ax1.set_xticklabels(xticklabels)
ax1.set_yticks(range(11))
ax1.set_yticklabels(yticklabels)

# Adjust layout
plt.tight_layout()

# Show the plot
plt.savefig(results_path + "Figure 2a: observed joint distribution of RGC vs AoU WES constraint predictions")
plt.show()

fig, ax2 = plt.subplots(1, figsize=(7, 6))

# Plot chi-squared statistic
ax2 = sns.heatmap(chi_sqr, cmap='Blues', annot=False, fmt=".3f", cbar=True, ax=ax2)
ax2.set_title('$\chi^2$ for Observed vs Expected Joint Distribution', y=1, x=0.55, fontsize=14)
ax2.set_xlabel('RGC Constraint Probability', fontsize=12)
ax2.set_ylabel('AoU Constraint Probability', fontsize=12)
ax2.invert_yaxis()
ax2.set_xticks(range(11))
ax2.set_xticklabels(xticklabels)
ax2.set_yticks(range(11))
ax2.set_yticklabels(yticklabels)

# Adjust layout
plt.tight_layout()

# Show the plot
plt.savefig(results_path + "Figure 2b: chi-square statistics of RGC vs AoU WES constraint predictions")
plt.show()

### HMM vs MTR and gnomAD Missense/LoF zscore

In [ ]:
# Function to calculate the overlap proportion for a single row
def calculate_overlap(row, chr_pos_dict):
    positions = chr_pos_dict[row['chr']]
    return np.sum((row['start'] <= positions) & (positions <= row['end'])) / row['length']

# Filter for the positions with probability > 0.8 of observing a 0
pos_over_80_dict = {'chr' + str(chromnum):
                    predictions_df[(predictions_df['chr'] == 'chr' + str(chromnum)) & (predictions_df['prob_0'] > 0.8)]['pos'].to_numpy()
                    for chromnum in range(1,23)}


# Apply the function to each row
gene_constraint_df['proportion_over_80'] = gene_constraint_df.apply(lambda row: calculate_overlap(row, pos_over_80_dict), axis=1)

# Filter for the positions with probability > 0.6 of observing a 0
pos_over_60_dict = {'chr' + str(chromnum):
                    predictions_df[(predictions_df['chr'] == 'chr' + str(chromnum)) & (predictions_df['prob_0'] > 0.6)]['pos'].to_numpy()
                    for chromnum in range(1,23)}

# Apply the function to each row
gene_constraint_df['proportion_over_60'] = gene_constraint_df.apply(lambda row: calculate_overlap(row, pos_over_60_dict), axis=1)

# Filter for the positions with probability > 0.5 of observing a 0
pos_over_50_dict = {'chr' + str(chromnum):
                    predictions_df[(predictions_df['chr'] == 'chr' + str(chromnum)) & (predictions_df['prob_0'] > 0.5)]['pos'].to_numpy()
                    for chromnum in range(1,23)}

# Apply the function to each row
gene_constraint_df['proportion_over_50'] = gene_constraint_df.apply(lambda row: calculate_overlap(row, pos_over_50_dict), axis=1)

# Display the result
gene_constraint_df

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['mis.z_score', 'proportion_over_80'])

X = filtered_df['mis.z_score']
X = sm.add_constant(X)
y = filtered_df['proportion_over_80']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_80'], gene_constraint_df['mis.z_score'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.8$')
# ax.set_ylabel('Missense Tolerance Ratio z-score')
ax.set_ylabel('Z-score of observed vs expected')
ax.set_xlim([0, 0.5])  # Adjust xlim if needed
ax.set_ylim([-10, 10])   # Adjust ylim if needed
# ax.set_title('Constraint Predictions vs MTR per gene')
ax.set_title('HMM Constraint Predictions vs gnomAD Z-score per gene')
fig.colorbar(density, label='Number of points per pixel')
ax.text(0.3, -7, '$R^2=0.186$', fontsize=10)
plt.show()

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['lof.z_score', 'proportion_over_60'])

X = filtered_df['lof.z_score']
X = sm.add_constant(X)
y = filtered_df['proportion_over_60']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_60'], gene_constraint_df['lof.z_score'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.6$')
# ax.set_ylabel('Missense Tolerance Ratio z-score')
ax.set_ylabel('LoF Z-score')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([-10, 10])   # Adjust ylim if needed
# ax.set_title('Constraint Predictions vs MTR per gene')
ax.set_title('HMM Constraint Predictions per gene vs gnomAD v4 ')
# fig.colorbar(density, label='Number of points per pix/el')
ax.text(0.4, -7, '$R^2=0.110$', fontsize=10)
plt.show()

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['mis.z_score', 'proportion_over_60'])

X = filtered_df['mis.z_score']
X = sm.add_constant(X)
y = filtered_df['proportion_over_60']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()

print(model.summary())

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_60'], gene_constraint_df['mis.z_score'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.6$')
# ax.set_ylabel('Missense Tolerance Ratio z-score')
ax.set_ylabel('Missense Z-score')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([-10, 10])   # Adjust ylim if needed
# ax.set_title('Constraint Predictions vs MTR per gene')
ax.set_title('HMM vs gnomAD v4 Constraint per gene')
# fig.colorbar(density, label='Number of points per pix/el')
ax.text(0.4, -7, '$R^2=0.207$', fontsize=10)
plt.savefig(results_path + "Figure 3a: HMM vs gnomAD v4 constraint per gene")
plt.show()

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['MTR', 'proportion_over_60'])

X = filtered_df['MTR']
X = sm.add_constant(X)
y = filtered_df['proportion_over_60']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()

print(model.summary())

In [11]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_60'], gene_constraint_df['MTR'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.6$')
ax.set_ylabel('Missense Tolerance Ratio')
# ax.set_ylabel('Z-score of observed vs expected')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([0.7, 1.1])   # Adjust ylim if needed
ax.set_title('HMM Constraint vs MTR per gene')
# ax.set_title('HMM Constraint Predictions vs gnomAD Z-score per gene')
# fig.colorbar(density, label='Number of points per pixel')
ax.text(0.5, 1.05, '$R^2=0.152$', fontsize=10)
plt.savefig(results_path + "Figure 3b: HMM vs MTR per gene")
plt.show()

### threshold 0.5

In [ ]:
# Function to calculate the overlap proportion for a single row
def calculate_overlap(row, chr_pos_dict):
    positions = chr_pos_dict[row['chr']]
    return np.sum((row['start'] <= positions) & (positions <= row['end'])) / row['length']

# Filter for the positions with probability > 0.5 of observing a 0
pos_over_50_dict = {'chr' + str(chromnum):
                    predictions_df[(predictions_df['chr'] == 'chr' + str(chromnum)) & (predictions_df['prob_0'] > 0.5)]['pos'].to_numpy()
                    for chromnum in range(1,23)}

# Apply the function to each row
gene_constraint_df['proportion_over_50'] = gene_constraint_df.apply(lambda row: calculate_overlap(row, pos_over_50_dict), axis=1)

# Display the result
gene_constraint_df

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['mis.z_score', 'proportion_over_50'])

X = filtered_df['mis.z_score']
X = sm.add_constant(X)
y = filtered_df['proportion_over_50']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()
rsquared = model.rsquared

print(model.summary())

In [14]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_50'], gene_constraint_df['mis.z_score'], cmap=white_viridis)
ax.set_xlabel('Fraction of coding bases with $\mathbb{P}(0) > 0.5$')
# ax.set_ylabel('Missense Tolerance Ratio z-score')
ax.set_ylabel('Missense Z-score')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([-10, 10])   # Adjust ylim if needed
# ax.set_title('Constraint Predictions vs MTR per gene')
ax.set_title('HMM Constraint Proportion vs\n gnomAD v4 Missense Z-score per gene')
# fig.colorbar(density, label='Number of points per pix/el')
ax.text(0.4, -7, f'$R^2={round(rsquared,3)}$', fontsize=10)
plt.savefig(results_path + "Figure 3a: HMM vs gnomAD missense z-score per gene")
plt.show()

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['lof.z_score', 'proportion_over_50'])

X = filtered_df['lof.z_score']
X = sm.add_constant(X)
y = filtered_df['proportion_over_50']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()
rsquared = model.rsquared

print(model.summary())

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_50'], gene_constraint_df['lof.z_score'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.5$')
ax.set_ylabel('LoF Z-score')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([-10, 10])   # Adjust ylim if needed
# ax.set_title('Constraint Predictions vs MTR per gene')
ax.set_title('HMM Constraint Proportion vs gnomAD v4 LoF Z-score per gene')
# fig.colorbar(density, label='Number of points per pix/el')
ax.text(0.4, -7, f'$R^2={round(rsquared,3)}$', fontsize=10)
plt.savefig(results_path + "Figure 3b: HMM vs gnomAD LoF z-score per gene")
plt.show()

In [ ]:
# Drop rows with missing or NaN values
filtered_df = gene_constraint_df.dropna(subset=['MTR', 'proportion_over_50'])

X = filtered_df['MTR']
X = sm.add_constant(X)
y = filtered_df['proportion_over_50']

# Fit the logistic regression model
model = sm.OLS(y, X).fit()
rsquared = model.rsquared

print(model.summary())

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = fig.add_subplot(111, projection='scatter_density')
density = ax.scatter_density(gene_constraint_df['proportion_over_50'], gene_constraint_df['MTR'], cmap=white_viridis)
ax.set_xlabel('Proportion of gene with $\mathbb{P}(0) > 0.5$')
ax.set_ylabel('Missense Tolerance Ratio')
# ax.set_ylabel('Z-score of observed vs expected')
ax.set_xlim([0, 0.7])  # Adjust xlim if needed
ax.set_ylim([0.7, 1.1])   # Adjust ylim if needed
ax.set_title('HMM Constraint Proportion vs MTR per gene')
# ax.set_title('HMM Constraint Predictions vs gnomAD Z-score per gene')
# fig.colorbar(density, label='Number of points per pixel')
ax.text(0.5, 1.05, f'$R^2={round(rsquared,3)}$', fontsize=10)
plt.savefig(results_path + "Figure 3c: HMM vs MTR per gene")
plt.show()

## Compare to MTR at the transcript level

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# -----------------------
# SETTINGS
# -----------------------
THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]
CHROMS = [f"chr{i}" for i in range(1, 23)]
POINT_ALPHA = 0.8
POINT_SIZE = 18

# -----------------------
# 0) Transcript-level MTR
# -----------------------
tx_mtr = (constraint_df[["transcript", "gene", "MTR"]]
          .dropna(subset=["transcript", "MTR"])
          .drop_duplicates(subset=["transcript"])
          .rename(columns={"gene": "gene_name"}))
tx_keep_set = set(tx_mtr["transcript"])

# -----------------------
# 1) CDS intervals + cumulative offsets
# -----------------------
cds = gene_df[(gene_df["feature"] == "CDS") &
              (gene_df["gene_type"] == "protein_coding") &
              (gene_df["transcript"].isin(tx_keep_set))].copy()
cds = cds[cds["chr"].isin(CHROMS)].copy()

cds["start"] = cds["start"].astype(int)
cds["end"] = cds["end"].astype(int)

# sort exons in coding order
cds["sort_key"] = np.where(cds["strand"] == "+", cds["start"], -cds["start"])
cds = cds.sort_values(["transcript", "chr", "sort_key", "start", "end"]).drop(columns=["sort_key"])

cds["exon_len"] = (cds["end"] - cds["start"] + 1).astype(int)
cds["cum_before"] = cds.groupby("transcript")["exon_len"].cumsum().shift(fill_value=0).astype(int)

# keep only transcripts on one chromosome
valid_tx = set(cds.groupby("transcript")["chr"].nunique().loc[lambda s: s == 1].index)
cds = cds[cds["transcript"].isin(valid_tx)].copy()

# -----------------------
# 2) Predictions indexed per chromosome (ALL bases)
# -----------------------
pred = predictions_df[["chr", "pos", "prob_0"]].dropna().copy()
pred = pred[pred["chr"].isin(CHROMS)].copy()
pred["pos"] = pred["pos"].astype(int)
pred["prob_0"] = pred["prob_0"].astype(float)

pred_by_chr = {}
for c in CHROMS:
    sub = pred[pred["chr"] == c].sort_values("pos")
    pred_by_chr[c] = (sub["pos"].to_numpy(), sub["prob_0"].to_numpy())

# -----------------------
# 3) Map genomic bases -> unique (tx, codon_index, base_slot) and deduplicate
# -----------------------
rows = []
for tx, exons in cds.groupby("transcript", sort=False):
    strand = exons["strand"].iloc[0]
    chrom = exons["chr"].iloc[0]
    pos_arr, p0_arr = pred_by_chr.get(chrom, (None, None))
    if pos_arr is None or pos_arr.size == 0:
        continue

    for _, ex in exons.iterrows():
        s, e = int(ex["start"]), int(ex["end"])
        cb = int(ex["cum_before"])

        lo = np.searchsorted(pos_arr, s, side="left")
        hi = np.searchsorted(pos_arr, e, side="right")
        if hi <= lo:
            continue

        exon_pos = pos_arr[lo:hi]
        exon_p0 = p0_arr[lo:hi]

        if strand == "+":
            cds_offset = cb + (exon_pos - s)
        else:
            cds_offset = cb + (e - exon_pos)

        codon_index = (cds_offset // 3).astype(int) + 1
        base_slot = (cds_offset % 3).astype(int)

        rows.append(pd.DataFrame({
            "transcript": tx,
            "codon_index": codon_index,
            "base_slot": base_slot,
            "prob_0": exon_p0
        }))

base_map_all = pd.concat(rows, ignore_index=True)

# Deduplicate base slots to avoid overlap/double-counting artifacts
base_map_all = (base_map_all
                .groupby(["transcript", "codon_index", "base_slot"], as_index=False)["prob_0"]
                .max())

# -----------------------
# 4) Build transcript-level HMM metrics
# -----------------------

# Continuous transcript metric: mean prob_0 across ALL deduped coding bases
tx_cont = (base_map_all
           .groupby("transcript", as_index=False)
           .agg(hmm_cont=("prob_0", "mean"),
                n_bases=("base_slot", "count")))
tx_cont = tx_cont.merge(tx_mtr[["transcript", "gene_name", "MTR"]], on="transcript", how="inner")

def build_tx_binary(thr: float) -> pd.DataFrame:
    # Fraction of deduped coding bases with prob_0 > thr (base-level analogue of your gene proportion)
    txb = (base_map_all.assign(over=(base_map_all["prob_0"] > thr).astype(int))
           .groupby("transcript", as_index=False)
           .agg(hmm_bin=("over", "mean"),
                n_bases=("over", "size")))
    txb = txb.merge(tx_mtr[["transcript", "gene_name", "MTR"]], on="transcript", how="inner")
    return txb

def r2_of_hmm_on_mtr(df, hmm_col="hmm", mtr_col="MTR"):
    d = df[[hmm_col, mtr_col]].dropna()
    X = sm.add_constant(d[mtr_col].to_numpy(float))
    y = d[hmm_col].to_numpy(float)
    return sm.OLS(y, X).fit().rsquared



In [ ]:
def _ols_r2(y, x):
    """R^2 for OLS y ~ const + x."""
    d = np.isfinite(y) & np.isfinite(x)
    y = y[d].astype(float)
    x = x[d].astype(float)
    X = sm.add_constant(x)
    return sm.OLS(y, X).fit().rsquared

def _scatter_density_plot(x, y, title, xlabel, ylabel, xlim=None, ylim=None):
    fig = plt.figure(figsize=(5.5, 5.5))
    ax = fig.add_subplot(111, projection="scatter_density")
    ax.scatter_density(x, y, cmap=white_viridis)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    # R^2 annotation bottom-right (inset, not on the edge)
    r2 = _ols_r2(y=np.asarray(x), x=np.asarray(y))  # y = HMM metric (x-axis), x = MTR (y-axis)
    ax.text(0.70, 0.15, rf"$R^2$={r2:.3f}", transform=ax.transAxes,
            ha="left", va="bottom", fontsize=11)

    plt.show()


# -----------------------
# 5) Continuous plot
#    Requires: tx_cont dataframe with columns ['hmm_cont','MTR']
# -----------------------
_scatter_density_plot(
    x=tx_cont["hmm_cont"].to_numpy(float),
    y=tx_cont["MTR"].to_numpy(float),
    title="Transcript-level HMM (continuous) vs MTR",
    xlabel="Mean $\mathbb{P}(0)$ across coding bases",
    ylabel="Missense Tolerance Ratio",
    xlim=(0, 0.7),
    ylim=(0.0, 1.1)
)

# -----------------------
# 6) Binary threshold plots
#    build_tx_binary(thr) returns dataframe with columns ['hmm_bin','MTR']
# -----------------------
for thr in THRESHOLDS:
    txb = build_tx_binary(thr)

    _scatter_density_plot(
        x=txb["hmm_bin"].to_numpy(float),
        y=txb["MTR"].to_numpy(float),
        title=f"HMM Constraint Proportion vs MTR per transcript",
        xlabel="Fraction of coding bases with $\mathbb{P}(0) > " + f"{thr}$",
        ylabel="Missense Tolerance Ratio",
        xlim=(0, 0.7),
        ylim=(0.0, 1.1)
    )